# GetAround — Analyse exploratoire des retards

Objectif : Aider le Product Manager à décider d'un seuil minimum entre deux locations et du périmètre d'application (toutes les voitures ou Connect uniquement).

In [1]:
import pandas as pd
import plotly.express as px 
import plotly.graph_objects as go
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots

### Chargmeent des donnéest et nettoyage

In [2]:
data = pd.read_excel("../01_Data/get_around_delay_analysis.xlsx")

# On conserve uniquement les colonnes utiles
data = data[[
    'rental_id', 'car_id', 'checkin_type', 'state',
    'delay_at_checkout_in_minutes',
    'previous_ended_rental_id',
    'time_delta_with_previous_rental_in_minutes'
]]

print(f"Shape : {data.shape}")
data.head()

Shape : (21335, 7)


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000.0,363965.0,mobile,canceled,NaN,NaN,NaN
1,507750.0,269550.0,mobile,ended,-81.0,NaN,NaN
2,508131.0,359049.0,connect,ended,70.0,NaN,NaN
3,508865.0,299063.0,connect,canceled,NaN,NaN,NaN
4,511440.0,313932.0,mobile,ended,NaN,NaN,NaN


In [3]:
# Aperçu des valeurs manquantes
missing = data.isnull().sum().rename('valeurs manquantes').to_frame()
missing['%'] = (missing['valeurs manquantes'] / len(data) * 100).round(1)
missing

,valeurs manquantes,%
rental_id,25,0.1
car_id,25,0.1
checkin_type,25,0.1
state,25,0.1
delay_at_checkout_in_minutes,4989,23.4
previous_ended_rental_id,19494,91.4
time_delta_with_previous_rental_in_minutes,19494,91.4


- delay_at_checkout_in_minutes : manquant pour 23,4%: locations annulées ou données absentes
- previous_ended_rental_id et time_delta : renseignés uniquement pour les 1 841 locations consécutives sur la même voiture

### Global

In [4]:
# Répartition par type de checkin et statut
counts = data.groupby(['checkin_type', 'state']).size().reset_index(name='count')

fig = px.bar(
    counts,
    x='checkin_type', y='count', color='state',
    barmode='group',
    text='count',
    title='Répartition des locations par type de checkin et statut',
    labels={'checkin_type': 'Type de checkin', 'count': 'Nombre de locations', 'state': 'Statut'},
    color_discrete_map={'ended': '#2ecc71', 'canceled': '#e74c3c'}
)
fig.update_traces(textposition='outside')
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [5]:
# Taux d'annulation par type
cancel_rate = data.groupby('checkin_type').apply(
    lambda x: round(100 * (x['state'] == 'canceled').sum() / len(x), 1)
).rename('taux_annulation_%').reset_index()

print(cancel_rate.to_string(index=False))

checkin_type  taux_annulation_%
     connect               18.5
      mobile               14.5


/var/folders/d2/j007c_wj355g0r_h_j5t1vrc0000gn/T/ipykernel_38112/2204458408.py:2: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



taux d'annulation similaire entre connect et mobile

### Analyse des retards au checkout

In [6]:
delays = data['delay_at_checkout_in_minutes'].dropna()

pct_late  = round(100 * (delays > 0).sum() / len(delays), 1)
print(f"En retard  (delay > 0) : {pct_late}%")

En retard  (delay > 0) : 57.5%


In [7]:
# Distribution des retards (filtrée à [-300, 600] pour la lisibilité)
delays_filtered = delays[(delays >= -300) & (delays <= 600)]

fig = px.histogram(
    delays_filtered,
    nbins=80,
    title='Distribution des retards au checkout (en minutes)',
    labels={'value': 'Retard (min)', 'count': 'Nombre de locations'},
    color_discrete_sequence=['#3498db']
)
fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='À l\'heure')
fig.add_vline(x=delays.median(), line_dash='dot', line_color='orange',
              annotation_text=f'Médiane ({delays.median():.0f} min)')
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [8]:
# Retard médian par type de checkin
fig = px.box(
    data[data['delay_at_checkout_in_minutes'].between(-300, 600)],
    x='checkin_type', y='delay_at_checkout_in_minutes',
    color='checkin_type',
    title='Distribution des retards par type de checkin',
    labels={'checkin_type': 'Type de checkin', 'delay_at_checkout_in_minutes': 'Retard (min)'},
    color_discrete_map={'mobile': '#3498db', 'connect': '#e67e22'}
)
fig.add_hline(y=0, line_dash='dash', line_color='red')
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig.show()

In [9]:
# % de retard par type de checkin
for ct in ['mobile', 'connect']:
    sub = data[data['checkin_type'] == ct]['delay_at_checkout_in_minutes'].dropna()
    print(f"{ct:8s} | médiane={sub.median():.0f} min | moyenne={sub.mean():.0f} min | % en retard={100*(sub>0).sum()/len(sub):.1f}%")

mobile   | médiane=14 min | moyenne=87 min | % en retard=61.4%
connect  | médiane=-9 min | moyenne=-44 min | % en retard=42.9%


bservations
- 57.5% des chauffeurs rendent la voiture en retard
- Les voitures mobile ont un retard médian de 14 min vs -9 min pour Connect
- Connect est nettement plus ponctuel : le chauffeur ouvre la voiture seul, sans contrainte de rendez-vous

### Impact sur la location suivante

uniquement sur les locations qui avaient une location précédente sur la même voiture.

In [10]:
# Jointure : location suivante ← location précédente (terminée)
df_next = data[data['previous_ended_rental_id'].notna()].copy()
df_next['previous_ended_rental_id'] = df_next['previous_ended_rental_id'].astype(int)

prev_info = data[data['state'] == 'ended'][['rental_id', 'delay_at_checkout_in_minutes', 'checkin_type']].copy()
prev_info.columns = ['prev_rental_id', 'prev_delay', 'prev_checkin_type']

merged = df_next.merge(prev_info, left_on='previous_ended_rental_id', right_on='prev_rental_id', how='inner')

print(f"Locations avec une précédente : {merged.shape[0]}")

Locations avec une précédente : 1841


In [11]:
# Parmi ces locations, combien ont été impactées par le retard du précédent ?
impacted     = merged[merged['prev_delay'] > 0]
not_impacted = merged[merged['prev_delay'] <= 0]

print(f"Retard précédent > 0 (next driver impacté) : {len(impacted)} ({100*len(impacted)/len(merged):.1f}%)")
print(f"  → dont location suivante annulée  : {(impacted['state']=='canceled').sum()}")
print(f"  → dont location suivante terminée : {(impacted['state']=='ended').sum()} (chauffeur a attendu)")
print()
print(f"Retard précédent ≤ 0 (pas d'impact) : {len(not_impacted)} ({100*len(not_impacted)/len(merged):.1f}%)")

Retard précédent > 0 (next driver impacté) : 873 (47.4%)
  → dont location suivante annulée  : 106
  → dont location suivante terminée : 767 (chauffeur a attendu)

Retard précédent ≤ 0 (pas d'impact) : 856 (46.5%)


In [12]:
# Distribution du délai entre deux locations consécutives
fig = px.histogram(
    merged,
    x='time_delta_with_previous_rental_in_minutes',
    nbins=50,
    color='checkin_type',
    barmode='overlay',
    opacity=0.7,
    title='Distribution du délai entre deux locations consécutives',
    labels={
        'time_delta_with_previous_rental_in_minutes': 'Délai (min)',
        'checkin_type': 'Type de checkin'
    },
    color_discrete_map={'mobile': '#3498db', 'connect': '#e67e22'}
)
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [13]:
# Retard précédent vs délai prévu — scatter
plot_data = merged[
    (merged['prev_delay'].between(-200, 600)) &
    (merged['time_delta_with_previous_rental_in_minutes'] <= 720)
].copy()
plot_data['impact'] = plot_data['prev_delay'] > plot_data['time_delta_with_previous_rental_in_minutes']

fig = px.scatter(
    plot_data,
    x='time_delta_with_previous_rental_in_minutes',
    y='prev_delay',
    color='impact',
    title='Retard précédent vs délai prévu entre locations',
    labels={
        'time_delta_with_previous_rental_in_minutes': 'Délai prévu (min)',
        'prev_delay': 'Retard location précédente (min)',
        'impact': 'Chevauchement'
    },
    color_discrete_map={True: '#e74c3c', False: '#2ecc71'},
    opacity=0.5
)
# Ligne y = x : si le retard > délai prévu → chevauchement
fig.add_shape(type='line', x0=0, y0=0, x1=720, y1=720,
              line=dict(color='black', dash='dash'))
fig.update_layout(plot_bgcolor='white', paper_bgcolor='white')
fig.show()

### Simulation de seuils
Pour chaque seuil, on calcule :
- Locations bloquées : créneaux qui auraient été retirés du catalogue
- Cas problématiques résolus : retards précédents que le seuil aurait absorbés

In [14]:
thresholds = [0, 30, 60, 120, 240, 480]
results = []

for scope in ['Tous', 'Connect uniquement']:
    if scope == 'Connect uniquement':
        subset = merged[merged['checkin_type'] == 'connect']
        total  = data[data['checkin_type'] == 'connect'].shape[0]
    else:
        subset = merged
        total  = data.shape[0]

    for t in thresholds:
        blocked = subset[subset['time_delta_with_previous_rental_in_minutes'] < t].shape[0]
        solved  = subset[
            (subset['prev_delay'] > 0) &
            (subset['prev_delay'] > (t - subset['time_delta_with_previous_rental_in_minutes']))
        ].shape[0]
        results.append({
            'Scope': scope,
            'Seuil (min)': t,
            'Locations bloquées': blocked,
            '% locations bloquées': round(100 * blocked / total, 2),
            'Cas problématiques résolus': solved,
        })

df_results = pd.DataFrame(results)
df_results

,Scope,Seuil (min),Locations bloquées,% locations bloquées,Cas problématiques résolus
0,Tous,0,0,0.00,873
1,Tous,30,279,1.31,805
2,Tous,60,401,1.88,754
3,Tous,120,666,3.12,653
4,Tous,240,1001,4.69,482
5,Tous,480,1290,6.05,294
6,Connect uniquement,0,0,0.00,304
7,Connect uniquement,30,131,3.04,281
8,Connect uniquement,60,181,4.20,260
9,Connect uniquement,120,295,6.85,229


In [15]:
# Visualisation : trade-off bloquées vs résolues
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Locations bloquées', 'Cas problématiques résolus']
)

colors = {'Tous': '#3498db', 'Connect uniquement': '#e67e22'}

for scope in ['Tous', 'Connect uniquement']:
    sub = df_results[df_results['Scope'] == scope]
    fig.add_trace(
        go.Bar(x=sub['Seuil (min)'].astype(str), y=sub['Locations bloquées'],
               name=scope, marker_color=colors[scope],
               text=sub['Locations bloquées'], textposition='outside',
               showlegend=True),
        row=1, col=1
    )
    fig.add_trace(
        go.Bar(x=sub['Seuil (min)'].astype(str), y=sub['Cas problématiques résolus'],
               name=scope, marker_color=colors[scope],
               text=sub['Cas problématiques résolus'], textposition='outside',
               showlegend=False),
        row=1, col=2
    )

fig.update_layout(
    title='Trade-off : seuil minimum entre deux locations',
    barmode='group',
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis_title='Seuil (min)', xaxis2_title='Seuil (min)'
)
fig.show()

In [16]:
# Ratio efficacité : cas résolus / locations bloquées
df_results['Ratio (résolus / bloqués)'] = (
    df_results['Cas problématiques résolus'] /
    df_results['Locations bloquées'].replace(0, np.nan)
).round(2)

df_results[df_results['Seuil (min)'] > 0][[
    'Scope', 'Seuil (min)', 'Locations bloquées',
    'Cas problématiques résolus', 'Ratio (résolus / bloqués)'
]]

,Scope,Seuil (min),Locations bloquées,Cas problématiques résolus,Ratio (résolus / bloqués)
1,Tous,30,279,805,2.89
2,Tous,60,401,754,1.88
3,Tous,120,666,653,0.98
4,Tous,240,1001,482,0.48
5,Tous,480,1290,294,0.23
7,Connect uniquement,30,131,281,2.15
8,Connect uniquement,60,181,260,1.44
9,Connect uniquement,120,295,229,0.78
10,Connect uniquement,240,430,181,0.42
11,Connect uniquement,480,550,124,0.23


### Recommandation

| Seuil | Scope | Locations bloquées | Cas résolus | Ratio |
|-------|-------|-------------------|-------------|-------|
| 30 min | Tous | 279 (1.3%) | 805 | 2.88 |
| 60 min | Tous | 401 (1.9%) | 754 | 1.88 |
| 60 min | Connect | 181 (4.2%) | 260 | 1.44 |
| 120 min | Tous | 666 (3.1%) | 653 | 0.98 |

**Un seuil de 60 minutes sur tous les types de checkin** offre le meilleur équilibre :
- Il résout **754 cas problématiques** tout en bloquant seulement **1.9% des créneaux**
- Au-delà de 120 min, le ratio s'inverse : on bloque plus de locations qu'on ne résout de problèmes
- Appliquer le seuil aux seules voitures Connect serait insuffisant car les retards sont majoritairement sur le **parc mobile** (61.4% en retard vs 42.9% pour Connect)